# 9.2 均衡问题的三层来源：为什么名义 shape 均匀 ≠ 负载均匀

分核的第一课：**分配的计量单位错了，一切均衡都是假象**。如果按「名义基本块数」分核，三件事会让真实负载严重倾斜。

## 1. 第一层：shape 本身不齐（B、N 不整除）

核数、batch、head 数互不为倍数时，简单的 `B×N / 核数` 均分切不动：B=3、N=64、20 核，无论如何切，都有核分到 10 个 BN 组、有的分到 9 个——这是离散分配的固有锯齿。

## 2. 第二层：mask 让名义块变无效块

因果 mask 下，S1×S2 的块矩阵里**上三角（或指定 band 之外）的块根本不用算**：

```
S1 块 0:  有效 S2 块 1/8        ← 因果 mask 下, 越靠前的 Q 行有效块越少
S1 块 1:  有效 S2 块 2/8
...
S1 块 7:  有效 S2 块 8/8
```

若按名义块数分核，前面的核闲死、后面的核算死。[第7章 7.4](../07_performance_basics/07.04_balance.ipynb) 的 `GetCutBlockNums` 减去被遮盖块，处理的正是这一层。

## 3. 第三层：section——KV 轴上的变长分段（本章主角）

推理的真实形态是**变长**的：

```
batch 0:  shared prefix 1024 + 独有 256    →  有效 KV = 1280
batch 1:  shared prefix 1024 + 独有 3072   →  有效 KV = 4096
batch 2:  shared prefix 1024 + 独有 128    →  有效 KV = 1152
```

两条 KV「section」（共享前缀段、独有段）合起来才是该请求的有效计算量。用名义 `maxSeqLen` 均分，batch 2 的核在算空气、batch 1 的核在超载。

> **section 的工程含义**：分核循环的输入不是 `B、N、S1、S2` 四个标量，而是 per-batch 的有效区间数组（`actualSeqLengths / actualSeqLengthsKV`，以及 prefix 的分段描述）。**分核从「算术题」变成「带变长权重的一维装箱题」**。

## 4. 正确的计量单位：有效基本块数

三层来源归结为一个结论：分核的计量单位必须是**每个 (b, n, s1) 行的有效 S2 基本块数**：

```
weight(b, n, s1) = ceil(有效S2起点差 / s2Base)   # 该行真正要算的块数
CoreWeight        = Σ weight / 核数                # 每核期望负载
```

后续 9.3/9.4 的两类策略，都是在「怎么遍历 (b, n, s1) 才能把 weight 装箱装得匀」上做文章。

## 动手实验：三层不均衡的量化


In [ ]:
# 动手：三层不均衡来源的量化对比
import numpy as np

n_cores = 20
S2_BASE = 128

# ---- 名义均分（什么都不管）----
B, N, S1, S2_max = 3, 64, 512, 4096
nominal_blocks = B * N * (S1 // 128) * (S2_max // S2_BASE)
print(f"名义总块数: {nominal_blocks}, 名义每核: {nominal_blocks / n_cores:.0f}")

# ---- 第一层: 离散锯齿 ----
bn_groups = B * N
per_core = [bn_groups // n_cores + (1 if i < bn_groups % n_cores else 0) for i in range(n_cores)]
print(f"BN 组数 {bn_groups} 分 {n_cores} 核: 每核 {per_core[0]}~{per_core[-1]} 组"
      f" (锯齿 {1 - per_core[-1]/per_core[0]:.0%} 相对差)")

# ---- 第二层: 因果 mask 有效块 ----
s1_blocks = S1 // 128
s2_blocks = S2_max // S2_BASE
rows = np.arange(s1_blocks)
valid = (rows + 1) * s2_blocks // s1_blocks      # 因果: 第 i 行有效块数
print(f"因果 mask 后有效块占比: {valid.sum() / (s1_blocks * s2_blocks):.0%}")

# ---- 第三层: section 变长 ----
actual_kv = [1280, 4096, 1152]                    # 3 个 batch 的有效 KV
blocks_per_batch = [np.ceil(kv / S2_BASE) for kv in actual_kv]
print(f"各 batch 有效 S2 块: {[int(b) for b in blocks_per_batch]}")
print(f"若按 maxSeqLen={S2_max} 名义均分, 短 batch 的核算了 "
      f"{(1 - min(actual_kv)/S2_max):.0%} 的空气")


### 观察结论

三层叠加后，「名义每核块数」这个数字已经完全失去意义。**分核函数必须拿有效块数说话**——这正是第7章素材中 `ComputeSplitNBSeq` 用 `GetSInnerBlockNums` 逐行统计有效块、再贪心装箱的原因。

## 5. 从均衡到分核：一维装箱的贪心解

拿到所有行的 weight 后，问题变成：把 `B×N×S1` 行（每行有权重）切成连续的 20 段，使各段权重和尽量相等。贪心策略：

```
target = 总权重 / 核数
curWeight = 0
遍历每一行:
    curWeight += weight(行)
    if curWeight 超过 target 的容差:
        封存当前核, 开启下一核
```

贪心不保证最优（一维装箱是 NP-hard），但**对 FA 的权重分布足够好**，且计算量 O(行数) 可以在 Host Tilling 侧完成。9.3 和 9.4 的差别就在「遍历行的顺序」上。

## 本节要点

- 三层不均衡：shape 锯齿、mask 无效块、section 变长；
- **分核计量单位 = 每个 (b,n,s1) 行的有效 S2 基本块数**，不是名义 shape；
- section 的输入形态是 per-batch 有效区间数组，分核从算术题变装箱题；
- 一维装箱贪心在 Host 侧 O(行数) 完成，是后续两节两类策略的共同底座。

## 小测验

1. 分层的计量单位为什么必须选「有效基本块数」？名义 shape 均分会错在哪三层？
2. 什么是 section？带 section 的输入形态和固定 shape 有什么本质区别？
3. 一维装箱问题为什么用贪心而不是求最优解？
4. `weight(b, n, s1)` 的定义是什么？CoreWeight 怎么算？

>>> 参考答案见 [answer/09.02_answer.txt](answer/09.02_answer.txt)
